# Spanish↔English MT with embedding + uniformity regularization

This notebook keeps the training loop explicit:
- no `Seq2SeqTrainingArguments`
- no custom `Trainer` subclass
- plain `model.train()`, `loss.backward()`, `optimizer.step()`

`MODEL_NAME` below is set to **Helsinki-NLP/opus-mt-es-en** for Spanish → English.
For English → Spanish, switch to **Helsinki-NLP/opus-mt-en-es** and swap `SRC_LANG` / `TGT_LANG`.


In [27]:
import os
import math
import json
import random
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

# Optional metrics
try:
    from comet import download_model, load_from_checkpoint
    COMET_AVAILABLE = True
except Exception:
    COMET_AVAILABLE = False

try:
    from bert_score import score as bertscore_score
    BERTSCORE_AVAILABLE = True
except Exception:
    BERTSCORE_AVAILABLE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cuda


In [28]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version PyTorch Wants:", torch.version.cuda)
print("Device Count:", torch.cuda.device_count())

PyTorch Version: 2.11.0+cu128
CUDA Available: True
CUDA Version PyTorch Wants: 12.8
Device Count: 1


In [29]:

# -------------------------
# Config
# -------------------------
MODEL_NAME = "Helsinki-NLP/opus-mt-es-en"
SRC_LANG = "es"
TGT_LANG = "en"

DATASET_NAME = "Helsinki-NLP/opus_books"
DATASET_CONFIG = "en-es"

MAX_SRC_LEN = 128
MAX_TGT_LEN = 128
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR = 5e-5
GRAD_CLIP = 1.0

# regularization strengths
LAMBDA_EMBED = 0.1
GAMMA_UNIFORM = 0.01
EMBED_LOSS_TYPE = "cosine"   # "cosine" or "l2"
UNIFORMITY_SAMPLE_SIZE = 256  # keeps L_uniform cheap


In [30]:

# -------------------------
# Data
# -------------------------
def load_translation_data(seed=42, test_size=0.1, val_size=0.1):
    ds = load_dataset(DATASET_NAME, DATASET_CONFIG)

    # opus_books only has a train split, so we make val/test ourselves
    tmp = ds["train"].train_test_split(test_size=(test_size + val_size), seed=seed)
    train_hf = tmp["train"]

    tmp2 = tmp["test"].train_test_split(
        test_size=val_size / (test_size + val_size),
        seed=seed,
    )
    val_hf = tmp2["train"]
    test_hf = tmp2["test"]
    return train_hf, val_hf, test_hf


class ParallelTextDataset(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]["translation"]
        return {"src": ex[SRC_LANG], "tgt": ex[TGT_LANG]}


def make_collate_fn(tokenizer, max_src_len=MAX_SRC_LEN, max_tgt_len=MAX_TGT_LEN):
    def collate_fn(batch):
        src_texts = [x["src"] for x in batch]
        tgt_texts = [x["tgt"] for x in batch]

        enc = tokenizer(
            src_texts,
            max_length=max_src_len,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        tgt = tokenizer(
            text_target=tgt_texts,
            max_length=max_tgt_len,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        labels = tgt["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100

        enc["labels"] = labels
        enc["src_text"] = src_texts
        enc["tgt_text"] = tgt_texts
        return enc

    return collate_fn


def move_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


In [31]:

# -------------------------
# Losses
# -------------------------
def embedding_loss_from_logits(logits, labels, token_embedding_weight, pad_token_id, loss_type="cosine"):
    """
    logits: [B, T, V]
    labels: [B, T]
    token_embedding_weight: [V, H]
    """
    probs = torch.softmax(logits, dim=-1)      # [B, T, V]
    e_hat = probs @ token_embedding_weight     # [B, T, H]

    gt_ids = labels.masked_fill(labels == -100, pad_token_id)
    e_gt = token_embedding_weight[gt_ids]      # [B, T, H]

    valid = labels != -100
    if valid.sum() == 0:
        return logits.new_tensor(0.0), e_hat

    if loss_type == "l2":
        per_tok = (e_hat - e_gt).pow(2).sum(dim=-1)
    elif loss_type == "cosine":
        per_tok = 1.0 - F.cosine_similarity(e_hat, e_gt, dim=-1)
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    return per_tok[valid].mean(), e_hat


def uniformity_loss(e_hat, labels, sample_size=UNIFORMITY_SAMPLE_SIZE):
    """
    Wang & Isola-style uniformity loss:

        log mean_{i != j} exp(-2 ||z_i - z_j||^2)

    Here we apply it to the predicted embeddings e_hat.
    """
    z = e_hat[labels != -100]  # [N, H]
    if z.size(0) < 2:
        return e_hat.new_tensor(0.0)

    if z.size(0) > sample_size:
        idx = torch.randperm(z.size(0), device=z.device)[:sample_size]
        z = z[idx]

    z = F.normalize(z, dim=-1)
    dist_sq = torch.cdist(z, z).pow(2)

    i, j = torch.triu_indices(z.size(0), z.size(0), offset=1, device=z.device)
    pair_terms = torch.exp(-2.0 * dist_sq[i, j])
    return torch.log(pair_terms.mean().clamp_min(1e-12))


def total_loss(logits, labels, token_embedding_weight, pad_token_id,
               lambda_embed=LAMBDA_EMBED, gamma_uniform=GAMMA_UNIFORM,
               embed_loss_type=EMBED_LOSS_TYPE):
    ce_loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100,
    )

    embed_loss, e_hat = embedding_loss_from_logits(
        logits=logits,
        labels=labels,
        token_embedding_weight=token_embedding_weight,
        pad_token_id=pad_token_id,
        loss_type=embed_loss_type,
    )
    uni_loss = uniformity_loss(e_hat, labels)

    loss = ce_loss + lambda_embed * embed_loss + gamma_uniform * uni_loss
    stats = {
        "loss": float(loss.detach().cpu()),
        "ce_loss": float(ce_loss.detach().cpu()),
        "embed_loss": float(embed_loss.detach().cpu()),
        "uniform_loss": float(uni_loss.detach().cpu()),
    }
    return loss, stats


In [ ]:

# -------------------------
# Metrics + evaluation
# -------------------------
def decode_texts(tokenizer, ids):
    return tokenizer.batch_decode(ids, skip_special_tokens=True)


@torch.no_grad()
def evaluate_model(model, dataloader, tokenizer, use_bertscore=False, comet_model=None):
    bleu_metric = BLEU()
    chrf_metric = CHRF(word_order=2)  # chrF++

    model.eval()
    sources, refs, hyps = [], [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        sources.extend(batch["src_text"])
        refs.extend(batch["tgt_text"])

        batch = move_to_device(batch, DEVICE)
        gen_ids = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=MAX_TGT_LEN,
        )
        hyps.extend(decode_texts(tokenizer, gen_ids))

    bleu = bleu_metric.corpus_score(hyps, [refs]).score
    chrfpp = chrf_metric.corpus_score(hyps, [refs]).score

    metrics = {
        "bleu": bleu,
        "chrfpp": chrfpp,
    }

    if use_bertscore and BERTSCORE_AVAILABLE:
        P, R, F1 = bertscore_score(hyps, refs, lang="en", rescale_with_baseline=True)
        metrics["bertscore_f1"] = float(F1.mean().item())

    if comet_model is not None:
        comet_data = [{"src": s, "mt": mt, "ref": r} for s, mt, r in zip(sources, hyps, refs)]
        comet_out = comet_model.predict(comet_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
        metrics["comet"] = float(comet_out.system_score)

    return metrics, sources, refs, hyps


In [33]:

# -------------------------
# Train loop
# -------------------------
def train_one_run(train_ds, val_ds, tokenizer, lambda_embed=0.1, gamma_uniform=0.01,
                  embed_loss_type="cosine", num_epochs=3, lr=5e-5, batch_size=16):
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.config.use_cache = False

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=make_collate_fn(tokenizer),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate_fn(tokenizer),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = num_epochs * len(train_loader)
    warmup_steps = max(1, int(0.1 * total_steps))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    history = []
    best_state = None
    best_score = -float("inf")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        model.train()
        running = 0.0

        for batch in tqdm(train_loader, desc="Training"):
            batch = move_to_device(batch, DEVICE)

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            logits = outputs.logits

            loss, stats = total_loss(
                logits=logits,
                labels=batch["labels"],
                token_embedding_weight=model.get_input_embeddings().weight,
                pad_token_id=tokenizer.pad_token_id,
                lambda_embed=lambda_embed,
                gamma_uniform=gamma_uniform,
                embed_loss_type=embed_loss_type,
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()

            running += stats["loss"]

        val_metrics, _, _, _ = evaluate_model(
            model,
            val_loader,
            tokenizer,
            use_bertscore=False,
            comet_model=None,
        )

        # choose best model by chrF++
        score = val_metrics["chrfpp"]
        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history.append({
            "epoch": epoch + 1,
            "train_loss": running / max(1, len(train_loader)),
            **val_metrics,
        })

        print(f"epoch {epoch+1}: train_loss={history[-1]['train_loss']:.4f}, "
              f"bleu={val_metrics['bleu']:.2f}, chrfpp={val_metrics['chrfpp']:.2f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def run_sweep(lambda_values=(0.0, 0.1, 0.4, 1.6), gamma_uniform=0.01, embed_loss_type="cosine"): # 0.05
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_hf, val_hf, test_hf = load_translation_data()

    train_ds = ParallelTextDataset(train_hf)
    val_ds = ParallelTextDataset(val_hf)
    test_ds = ParallelTextDataset(test_hf)

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=make_collate_fn(tokenizer),
    )

    comet_model = None
    if COMET_AVAILABLE:
        try:
            comet_ckpt = download_model("Unbabel/wmt22-comet-da")
            comet_model = load_from_checkpoint(comet_ckpt)
        except Exception:
            comet_model = None

    results = []
    for lam in lambda_values:
        print(f"\\n=== lambda_embed={lam} ===")
        model, history = train_one_run(
            train_ds=train_ds,
            val_ds=val_ds,
            tokenizer=tokenizer,
            lambda_embed=lam,
            gamma_uniform=gamma_uniform,
            embed_loss_type=embed_loss_type,
            num_epochs=NUM_EPOCHS,
            lr=LR,
            batch_size=BATCH_SIZE,
        )

        test_metrics, sources, refs, hyps = evaluate_model(
            model,
            test_loader,
            tokenizer,
            use_bertscore=False,
            comet_model=comet_model,
        )

        row = {
            "lambda_embed": lam,
            "gamma_uniform": gamma_uniform,
            "embed_loss_type": embed_loss_type,
            **test_metrics,
            "history": history,
        }
        results.append(row)

        print("test metrics:", {k: v for k, v in test_metrics.items() if isinstance(v, (int, float))})

    return results

In [34]:
results = run_sweep()
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

c:\Users\KatherynZhou\AppData\Local\anaconda3\envs\translation\lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 4977.81it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\KatherynZhou\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\KatherynZhou\AppData\Local\anaconda3\envs\translation\lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


\n=== lambda_embed=0.0 ===
Epoch 1/3


Training: 100%|██████████| 4674/4674 [05:12<00:00, 14.96it/s]


KeyboardInterrupt: 

In [35]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-es-en")

for text in ["big", "large", "He was sad", "He was unhappy"]:
    print(text, "->", tok.tokenize(text))

big -> ['▁', 'big']
large -> ['▁la', 'rge']
He was sad -> ['▁He', '▁wa', 's', '▁sa', 'd']
He was unhappy -> ['▁He', '▁wa', 's', '▁un', 'ha', 'ppy']
